In [1]:
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

print("Library berhasil dimuat!")

Library berhasil dimuat!


In [2]:
import os
import glob
import pandas as pd

# 1. Tentukan Path Asal dan Folder Tujuan
path_asal = r"C:\Users\Balqoz\skripsiku\Dataset Asli"
folder_tujuan = r"C:\Users\Balqoz\skripsiku\Hasil ExG"

# Buat folder tujuan secara otomatis jika belum ada
os.makedirs(folder_tujuan, exist_ok=True)

# Cari semua file Excel (.xlsx)
files = glob.glob(os.path.join(path_asal, "*.xlsx"))
print(f"Step 1 Selesai: Berhasil menemukan {len(files)} file Excel di folder asal.")

Step 1 Selesai: Berhasil menemukan 16 file Excel di folder asal.


In [3]:
def proses_dan_hitung_data(file_path):
    # a. Baca data mentah
    df_raw = pd.read_excel(file_path, header=None)
    
    # b. Ekstrak R, G, B, IR menggunakan Regex
    pattern = r"R=\s*(\d+)\s*G=\s*([\d.]+)\s*B=\s*([\d.]+)\s*(\d+)\s*%"
    extracted = df_raw[0].astype(str).str.extract(pattern)
    
    df = pd.DataFrame()
    df["R"] = pd.to_numeric(extracted[0])
    df["G"] = pd.to_numeric(extracted[1])
    df["B"] = pd.to_numeric(extracted[2])
    df["IR"] = pd.to_numeric(extracted[3])
    df.dropna(inplace=True)

    # c. Normalisasi (r, g, b)
    df["r"] = df["R"] / 255
    df["g"] = df["G"] / 255
    df["b"] = df["B"] / 255

    # d. Hitung Excess Green (ExG)
    df["ExG"] = (2 * df["g"]) - df["r"] - df["b"]

    # e. Hitung Klorofil Sementara
    df["A645"] = df["IR"] / 100
    df["A663"] = df["IR"] / 95
    df["A665"] = df["IR"] / 98
    df["A669"] = df["IR"] / 102
    df["Chl_A"] = (12.7 * df["A663"]) - (2.69 * df["A645"])
    df["Chl_B"] = (22.9 * df["A645"]) - (4.68 * df["A669"])
    df["Chl_Total"] = (20.2 * df["A645"]) + (8.02 * df["A665"])
    
    return df

In [4]:
data_list = []

print("Mulai memproses, merata-ratakan data, dan menyimpan file...\n")
for file in files:
    nama_asli = os.path.splitext(os.path.basename(file))[0]
    perlakuan_name = nama_asli.upper() 
    
    # 1. Ambil data mentah detikan dari fungsi Cell 3
    df_raw_processed = proses_dan_hitung_data(file)
    
    # 2. HITUNG RATA-RATA (Agar ringkas 1 baris per file seperti Tito)
    df_mean = df_raw_processed.mean().to_frame().T
    
    # 3. Tambahkan kolom Perlakuan dan Pengambilan (misal kita ambil angka dari nama file)
    df_mean.insert(0, "Perlakuan", perlakuan_name)
    
    # Mengambil nomor urut dari nama file untuk kolom Pengambilan
    nomor_file = "".join(filter(str.isdigit, nama_asli))
    df_mean.insert(1, "Pengambilan", int(nomor_file) if nomor_file else 1)
    
    # 4. Sesuaikan nama kolom agar mirip milik Tito
    df_mean = df_mean.rename(columns={
        "ExG": "Excess_Green",
        "IR": "IR_Intensity (%)"
    })
    
    # Urutkan kolom
    kolom_rapi = [
        "Perlakuan", "Pengambilan", "R", "G", "B", "IR_Intensity (%)", 
        "r", "g", "b", "Excess_Green", 
        "A645", "A663", "A665", "A669", 
        "Chl_A", "Chl_B", "Chl_Total"
    ]
    df_final = df_mean[kolom_rapi]
    
    # Simpan file Excel individu (isinya 1 baris rata-rata yang rapi)
    nama_file_simpan = f"Hasil_ExG_{nama_asli}.xlsx"
    path_simpan = os.path.join(folder_tujuan, nama_file_simpan)
    df_final.to_excel(path_simpan, index=False)
    
    print(f"  -> Berhasil merangkum & menyimpan: {nama_file_simpan}")
    data_list.append(df_final)

print("\nStep 3 Selesai: 16 File rangkuman sukses tersimpan!")

Mulai memproses, merata-ratakan data, dan menyimpan file...

  -> Berhasil merangkum & menyimpan: Hasil_ExG_HH01.xlsx
  -> Berhasil merangkum & menyimpan: Hasil_ExG_HH02.xlsx
  -> Berhasil merangkum & menyimpan: Hasil_ExG_HH03.xlsx
  -> Berhasil merangkum & menyimpan: Hasil_ExG_HH04.xlsx
  -> Berhasil merangkum & menyimpan: Hasil_ExG_HH11.xlsx
  -> Berhasil merangkum & menyimpan: Hasil_ExG_HH12.xlsx
  -> Berhasil merangkum & menyimpan: Hasil_ExG_HH13.xlsx
  -> Berhasil merangkum & menyimpan: Hasil_ExG_HH14.xlsx
  -> Berhasil merangkum & menyimpan: Hasil_ExG_HH21.xlsx
  -> Berhasil merangkum & menyimpan: Hasil_ExG_HH22.xlsx
  -> Berhasil merangkum & menyimpan: Hasil_ExG_HH23.xlsx
  -> Berhasil merangkum & menyimpan: Hasil_ExG_HH24.xlsx
  -> Berhasil merangkum & menyimpan: Hasil_ExG_HH31.xlsx
  -> Berhasil merangkum & menyimpan: Hasil_ExG_HH32.xlsx
  -> Berhasil merangkum & menyimpan: Hasil_ExG_HH33.xlsx
  -> Berhasil merangkum & menyimpan: Hasil_ExG_HH34.xlsx

Step 3 Selesai: 16 File ra

In [5]:
# Gabungkan ke-16 baris dari setiap file
df_all = pd.concat(data_list, ignore_index=True)

# Siapkan Fitur Input (X) untuk Machine Learning
X = df_all[["r", "g", "b", "Excess_Green"]]

print(f"Step 4 Selesai: Data berhasil digabungkan!")
print(f"Total Keseluruhan Baris Data saat ini: {df_all.shape[0]} baris (Sama seperti Tito).")
print("\nBerikut adalah tampilan data rata-rata Anda:")

# Munculkan tabel interaktif 16 baris Anda
df_all

Step 4 Selesai: Data berhasil digabungkan!
Total Keseluruhan Baris Data saat ini: 16 baris (Sama seperti Tito).

Berikut adalah tampilan data rata-rata Anda:


,Perlakuan,Pengambilan,R,G,B,IR_Intensity (%),r,g,b,Excess_Green,A645,A663,A665,A669,Chl_A,Chl_B,Chl_Total
0,HH01,1,148.693333,150.792000,117.351200,71.880000,0.583111,0.591341,0.460201,0.139370,0.718800,0.756632,0.733469,0.704706,7.675649,13.162496,20.402184
1,HH02,2,143.643564,147.645545,111.587129,89.643564,0.563308,0.579002,0.437597,0.157100,0.896436,0.943616,0.914730,0.878858,9.572517,16.415319,25.444137
2,HH03,3,143.641026,146.476923,109.260000,69.858974,0.563298,0.574419,0.428471,0.157070,0.698590,0.735358,0.712847,0.684892,7.459835,12.792411,19.828543
3,HH04,4,130.722222,136.500000,92.551389,68.111111,0.512636,0.535294,0.362947,0.195005,0.681111,0.716959,0.695011,0.667756,7.273191,12.472346,19.332435
4,HH11,11,151.890244,152.248780,114.314146,75.865854,0.595648,0.597054,0.448291,0.150169,0.758659,0.798588,0.774141,0.743783,8.101275,13.892377,21.533516
5,HH12,12,146.933333,150.376000,116.044267,74.973333,0.576209,0.589710,0.455076,0.148135,0.749733,0.789193,0.765034,0.735033,8.005968,13.728940,21.280186
6,HH13,13,145.550562,148.325843,114.868989,68.685393,0.570787,0.581670,0.450467,0.142087,0.686854,0.723004,0.700871,0.673386,7.334515,12.577508,19.495438
7,HH14,14,153.057377,154.822951,122.856721,70.090164,0.600225,0.607149,0.481791,0.132282,0.700902,0.737791,0.715206,0.687158,7.484523,12.834746,19.894163
8,HH21,21,152.852941,154.929412,123.797059,91.225490,0.599423,0.607566,0.485479,0.130231,0.912255,0.960268,0.930872,0.894368,9.741442,16.704997,25.893145
9,HH22,22,150.090909,153.772727,124.913182,93.681818,0.588592,0.603030,0.489856,0.127613,0.936818,0.986124,0.955937,0.918449,10.003739,17.154794,26.590341
